# YOLOv11 — Military Vehicle Detection (Aerial Imagery)

Train a YOLOv11 model to detect **Military Vehicles** and **Civilian Vehicles** in satellite/aerial images.

**Dataset**: MVRSD — ~2,260 images (640×640, 0.3m resolution) with 32K+ annotations.

---

## 1. Setup

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload Data

Upload `data/merged/` to Colab. Options:
- **Google Drive**: Mount drive and copy, or
- **Direct upload**: Zip `data/merged/` locally, upload, unzip

In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive to local storage (faster I/O)
!cp -r /content/drive/MyDrive/hack_ai/data/merged /content/data/merged

In [ ]:
# Option B: Upload a zip file
# from google.colab import files
# uploaded = files.upload()  # upload merged.zip
# !mkdir -p /content/data && unzip -q merged.zip -d /content/data/

In [ ]:
# Quick sanity check
!ls /content/data/merged/images/train/ | head -5
!echo '---'
!wc -l /content/data/merged/labels/train/*.txt | tail -1
!cat /content/data/merged/dataset.yaml

## 3. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")

results = model.train(
    data="/content/data/merged/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,        # Use 8 if OOM on free-tier T4
    patience=20,     # Early stopping
    project="/content/runs",
    name="military_v1",
    exist_ok=True,
)

## 4. Validate

In [ ]:
metrics = model.val()

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"\nPer-class AP50:")
for i, name in enumerate(['Military Vehicle', 'Civilian Vehicle']):
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")

## 5. Inference Demo

In [ ]:
import glob
from IPython.display import display, Image

val_images = sorted(glob.glob("/content/data/merged/images/val/*.jpg"))[:6]

results = model.predict(val_images, save=True, project="/content/runs", name="predict_demo", exist_ok=True)

for r in results:
    display(Image(filename=r.save_dir / r.path.split('/')[-1], width=400))

## 6. Export Model

In [ ]:
# Export to ONNX for deployment
model.export(format="onnx")

print("\nExported files:")
!ls -lh /content/runs/military_v1/weights/

## 7. Download Weights

In [ ]:
from google.colab import files

# Download best weights
files.download("/content/runs/military_v1/weights/best.pt")

# Optionally copy to Drive
# !cp -r /content/runs/military_v1/ /content/drive/MyDrive/hack_ai/runs/

## 8. Confusion Matrix & Analysis

In [ ]:
from IPython.display import display, Image

# Training artifacts
run_dir = "/content/runs/military_v1"
for plot in ["confusion_matrix.png", "results.png", "PR_curve.png", "F1_curve.png"]:
    path = f"{run_dir}/{plot}"
    try:
        print(f"\n{plot}")
        display(Image(filename=path, width=600))
    except FileNotFoundError:
        print(f"  (not found)")